# SEC EDGAR 원본 데이터 확인 및 진단

In [1]:
import sys
import pandas as pd
import requests
import json

sys.path.append(r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\US_Market\collect")

from sec_data_pipeline.valuation.integrated_financial_analyzer_mysql_fixed import IntegratedFinancialAnalyzer
from DATA.stock_invest_function import get_db_host

db_info = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",
    "database": "investar",
}

analyzer = IntegratedFinancialAnalyzer(db_info=db_info, wrds_conn_str="여기에_WRDS_주소")
headers = {"User-Agent": "Hoyoung Research <stox1224@gmail.com>"}

## 1. EDGAR Company Facts API 직접 호출

In [2]:
# GOOG의 CIK 번호
cik = "0001652044"
url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"

response = requests.get(url, headers=headers)
company_facts = response.json()

print("Company Name:", company_facts.get('entityName'))
print("CIK:", company_facts.get('cik'))
print("\nAvailable fact categories:")
print(list(company_facts.get('facts', {}).keys()))

Company Name: Alphabet Inc.
CIK: 1652044

Available fact categories:
['dei', 'us-gaap']


## 2. Revenue 원본 데이터 확인

In [3]:
# US-GAAP의 Revenues 항목 찾기
us_gaap_facts = company_facts.get('facts', {}).get('us-gaap', {})

# Revenue 관련 항목들 찾기
revenue_keys = [key for key in us_gaap_facts.keys() if 'revenue' in key.lower()]
print("Revenue 관련 항목들:")
for key in revenue_keys[:10]:
    print(f"  - {key}")

# Revenues 항목의 데이터 확인
if 'Revenues' in us_gaap_facts:
    revenues_data = us_gaap_facts['Revenues']
    print("\n\nRevenues 항목 구조:")
    print("Units:", list(revenues_data.get('units', {}).keys()))
    
    # USD 단위 데이터
    usd_data = revenues_data.get('units', {}).get('USD', [])
    print(f"\nUSD 데이터 개수: {len(usd_data)}")
    
    # DataFrame으로 변환
    df_revenue = pd.DataFrame(usd_data)
    print("\nRevenue 원본 데이터 컬럼:")
    print(df_revenue.columns.tolist())
    
    # 최근 데이터만 필터링 (2024년 이후)
    df_revenue['end'] = pd.to_datetime(df_revenue['end'])
    df_recent = df_revenue[df_revenue['end'] >= '2024-01-01'].copy()
    df_recent = df_recent.sort_values('end')
    
    print("\n\n2024년 이후 Revenue 원본 데이터:")
    print(df_recent[['end', 'val', 'accn', 'fy', 'fp', 'form', 'filed']].to_string())

Revenue 관련 항목들:
  - ContractWithCustomerLiabilityRevenueRecognized
  - CostOfRevenue
  - DeferredRevenueCurrent
  - DeferredRevenueNoncurrent
  - DeferredRevenueRevenueRecognized1
  - IncreaseDecreaseInDeferredRevenue
  - RevenueFromContractWithCustomerExcludingAssessedTax
  - RevenueRemainingPerformanceObligation
  - Revenues
  - RevenueNotFromContractWithCustomer


Revenues 항목 구조:
Units: ['USD']

USD 데이터 개수: 71

Revenue 원본 데이터 컬럼:
['start', 'end', 'val', 'accn', 'fy', 'fp', 'form', 'filed', 'frame']


2024년 이후 Revenue 원본 데이터:
          end           val                  accn    fy  fp  form       filed
61 2024-06-30  165281000000  0001652044-25-000062  2025  Q2  10-Q  2025-07-24
62 2024-06-30   84742000000  0001652044-25-000062  2025  Q2  10-Q  2025-07-24
63 2024-09-30  253549000000  0001652044-25-000091  2025  Q3  10-Q  2025-10-30
64 2024-09-30   88268000000  0001652044-25-000091  2025  Q3  10-Q  2025-10-30
65 2024-12-31  350018000000  0001652044-26-000018  2025  FY  10-K  2026-02-0

## 3. 분기별 데이터 상세 분석

In [4]:
# 10-Q (분기보고서)와 10-K (연간보고서) 구분
if 'Revenues' in us_gaap_facts:
    usd_data = us_gaap_facts['Revenues'].get('units', {}).get('USD', [])
    df_revenue = pd.DataFrame(usd_data)
    df_revenue['end'] = pd.to_datetime(df_revenue['end'])
    
    # 2024-2025년 데이터만
    df_recent = df_revenue[df_revenue['end'] >= '2024-01-01'].copy()
    df_recent = df_recent.sort_values('end')
    
    print("Form별 데이터 (10-Q: 분기보고서, 10-K: 연간보고서):")
    print(df_recent[['end', 'val', 'fy', 'fp', 'form', 'filed']].to_string())
    
    print("\n\n분기별 그룹화:")
    df_recent['year'] = df_recent['end'].dt.year
    df_recent['quarter'] = df_recent['end'].dt.quarter
    
    for (year, quarter), group in df_recent.groupby(['year', 'quarter']):
        print(f"\n{year} Q{quarter}:")
        print(group[['end', 'val', 'fp', 'form', 'filed']].to_string())
        
        # 같은 날짜에 여러 값이 있는지 확인
        if len(group) > 1:
            print(f"  ⚠ 주의: 같은 분기에 {len(group)}개의 데이터 존재!")

Form별 데이터 (10-Q: 분기보고서, 10-K: 연간보고서):
          end           val    fy  fp  form       filed
61 2024-06-30  165281000000  2025  Q2  10-Q  2025-07-24
62 2024-06-30   84742000000  2025  Q2  10-Q  2025-07-24
63 2024-09-30  253549000000  2025  Q3  10-Q  2025-10-30
64 2024-09-30   88268000000  2025  Q3  10-Q  2025-10-30
65 2024-12-31  350018000000  2025  FY  10-K  2026-02-05
66 2025-06-30  186662000000  2025  Q2  10-Q  2025-07-24
67 2025-06-30   96428000000  2025  Q2  10-Q  2025-07-24
68 2025-09-30  289007000000  2025  Q3  10-Q  2025-10-30
69 2025-09-30  102346000000  2025  Q3  10-Q  2025-10-30
70 2025-12-31  402836000000  2025  FY  10-K  2026-02-05


분기별 그룹화:

2024 Q2:
          end           val  fp  form       filed
61 2024-06-30  165281000000  Q2  10-Q  2025-07-24
62 2024-06-30   84742000000  Q2  10-Q  2025-07-24
  ⚠ 주의: 같은 분기에 2개의 데이터 존재!

2024 Q3:
          end           val  fp  form       filed
63 2024-09-30  253549000000  Q3  10-Q  2025-10-30
64 2024-09-30   88268000000  Q3  10-Q 

## 4. 현재 analyzer가 반환하는 데이터 확인

In [5]:
# analyzer를 통해 데이터 가져오기
result = analyzer.analyze('GOOG', headers=headers, table_name="us_fundq")

if result:
    final_df, cik, entity_name = result
    
    print(f"Entity: {entity_name} (CIK: {cik})")
    print(f"\nDataFrame shape: {final_df.shape}")
    print(f"Index range: {final_df.index.min()} ~ {final_df.index.max()}")
    
    print("\n2024년 이후 Revenue 데이터:")
    df_2024 = final_df[final_df.index >= '2024-01-01'].copy()
    
    if 'revenue' in df_2024.columns:
        print(df_2024[['revenue']].to_string())
    else:
        print("revenue 컬럼 없음")
        print("\n사용 가능한 컬럼:")
        print(df_2024.columns.tolist())

[GOOG] 통합 재무분석 시작
✓ Entity: Alphabet Inc. (CIK: 1652044)
✓ EDGAR 정규화 DF: 48 rows × 31 cols
✓ MySQL 연결 성공: 192.168.0.230:3307/investar
⚠ WRDS 쿼리 실패: Execution failed on sql 'SELECT * FROM us_fundq WHERE tic=%s': (1146, "Table 'investar.us_fundq' doesn't exist")
WRDS 데이터 없음 → EDGAR 그대로 반환
✓ EDGAR+WRDS 병합 DF: 48 rows × 31 cols
재무비율 계산 중...
  - 수익성 비율 계산...
  - 레버리지 비율 계산...
  - 유동성 비율 계산...
  - 효율성 비율 계산...
재무비율 계산 완료!
✓ 최종 DF (비율 포함): 48 rows × 47 cols
[GOOG] 통합 재무분석 종료
Entity: Alphabet Inc. (CIK: 1652044)

DataFrame shape: (48, 47)
Index range: 2012-12-31 00:00:00 ~ 2025-12-31 00:00:00

2024년 이후 Revenue 데이터:
                 revenue
date                    
2024-03-31  8.053900e+10
2024-06-30  8.474200e+10
2024-09-30  8.826800e+10
2024-12-31  8.826800e+10
2025-03-31  9.023400e+10
2025-06-30  9.023400e+10
2025-09-30  9.023400e+10
2025-12-31  9.023400e+10


## 5. 문제 진단 및 해결 방안

In [6]:
print("""
=== 문제 진단 체크리스트 ===

1. SEC EDGAR 원본 데이터 확인:
   - 같은 날짜(end)에 여러 개의 값이 있는가?
   - 10-Q (분기보고서)와 10-K (연간보고서) 모두 포함되어 있는가?
   - fp (fiscal period) 값이 Q1, Q2, Q3, FY로 구분되어 있는가?

2. 데이터 선택 로직 문제:
   - 같은 날짜에 여러 값이 있을 때 어떤 것을 선택하는가?
   - 10-Q의 누적값을 가져와야 하는데 10-K를 가져오고 있지 않은가?
   - frame 값 (원본/수정본)이 섞여 있지 않은가?

3. 해결 방안:
   A. 같은 날짜에 여러 값이 있으면:
      - 10-Q를 우선으로 선택
      - frame이 없는 것 (원본) 우선 선택
      - filed (제출일) 최신 것 선택
   
   B. fp (fiscal period) 활용:
      - Q1, Q2, Q3는 분기 누적값
      - FY는 연간 누적값
      - 각 분기 종료일에 해당하는 fp='Q1', 'Q2', 'Q3'만 선택
""")


=== 문제 진단 체크리스트 ===

1. SEC EDGAR 원본 데이터 확인:
   - 같은 날짜(end)에 여러 개의 값이 있는가?
   - 10-Q (분기보고서)와 10-K (연간보고서) 모두 포함되어 있는가?
   - fp (fiscal period) 값이 Q1, Q2, Q3, FY로 구분되어 있는가?

2. 데이터 선택 로직 문제:
   - 같은 날짜에 여러 값이 있을 때 어떤 것을 선택하는가?
   - 10-Q의 누적값을 가져와야 하는데 10-K를 가져오고 있지 않은가?
   - frame 값 (원본/수정본)이 섞여 있지 않은가?

3. 해결 방안:
   A. 같은 날짜에 여러 값이 있으면:
      - 10-Q를 우선으로 선택
      - frame이 없는 것 (원본) 우선 선택
      - filed (제출일) 최신 것 선택
   
   B. fp (fiscal period) 활용:
      - Q1, Q2, Q3는 분기 누적값
      - FY는 연간 누적값
      - 각 분기 종료일에 해당하는 fp='Q1', 'Q2', 'Q3'만 선택



## 6. 개선된 데이터 선택 로직

In [7]:
def select_best_quarterly_data(df_raw):
    """
    같은 날짜에 여러 데이터가 있을 때 최적의 데이터 선택
    
    우선순위:
    1. form='10-Q' 우선 (분기보고서)
    2. frame 없는 것 우선 (원본 데이터)
    3. filed 최신 것 선택
    """
    if df_raw.empty:
        return df_raw
    
    df = df_raw.copy()
    
    # end 날짜별로 그룹화
    result_list = []
    
    for end_date, group in df.groupby('end'):
        # 1순위: 10-Q 선택
        quarterly = group[group['form'] == '10-Q']
        if not quarterly.empty:
            group = quarterly
        
        # 2순위: frame 없는 것 선택
        no_frame = group[group['frame'].isna()]
        if not no_frame.empty:
            group = no_frame
        
        # 3순위: 최신 제출일 선택
        if 'filed' in group.columns:
            latest = group.sort_values('filed', ascending=False).iloc[0]
        else:
            latest = group.iloc[0]
        
        result_list.append(latest)
    
    return pd.DataFrame(result_list)


# 테스트
if 'Revenues' in us_gaap_facts:
    usd_data = us_gaap_facts['Revenues'].get('units', {}).get('USD', [])
    df_revenue = pd.DataFrame(usd_data)
    df_revenue['end'] = pd.to_datetime(df_revenue['end'])
    
    # 2024년 이후
    df_recent = df_revenue[df_revenue['end'] >= '2024-01-01'].copy()
    
    print("원본 데이터 (중복 포함):")
    print(f"총 {len(df_recent)}건")
    print(df_recent[['end', 'val', 'fp', 'form']].to_string())
    
    # 개선된 선택 로직 적용
    df_selected = select_best_quarterly_data(df_recent)
    df_selected = df_selected.sort_values('end')
    
    print("\n\n선택된 데이터 (중복 제거):")
    print(f"총 {len(df_selected)}건")
    print(df_selected[['end', 'val', 'fp', 'form', 'filed']].to_string())

원본 데이터 (중복 포함):
총 10건
          end           val  fp  form
61 2024-06-30  165281000000  Q2  10-Q
62 2024-06-30   84742000000  Q2  10-Q
63 2024-09-30  253549000000  Q3  10-Q
64 2024-09-30   88268000000  Q3  10-Q
65 2024-12-31  350018000000  FY  10-K
66 2025-06-30  186662000000  Q2  10-Q
67 2025-06-30   96428000000  Q2  10-Q
68 2025-09-30  289007000000  Q3  10-Q
69 2025-09-30  102346000000  Q3  10-Q
70 2025-12-31  402836000000  FY  10-K


선택된 데이터 (중복 제거):
총 6건
          end           val  fp  form       filed
61 2024-06-30  165281000000  Q2  10-Q  2025-07-24
63 2024-09-30  253549000000  Q3  10-Q  2025-10-30
65 2024-12-31  350018000000  FY  10-K  2026-02-05
66 2025-06-30  186662000000  Q2  10-Q  2025-07-24
68 2025-09-30  289007000000  Q3  10-Q  2025-10-30
70 2025-12-31  402836000000  FY  10-K  2026-02-05
